# Feature Engineering ML (Spark)
## 1. Imports & Setups

In [19]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [31]:
def create_ml_feature_store():
    """
    GOLD Layer – Feature Store ML
    Source : silver.trips_geospatial
    Output : iceberg.gold.ml_trip_features
    """

    print("🧠 Démarrage Feature Engineering ML")

    catalog = "iceberg"

    # ------------------------------------------------------------------
    # Namespace GOLD
    # ------------------------------------------------------------------
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {catalog}.gold")

    # ------------------------------------------------------------------
    # Source data
    # ------------------------------------------------------------------
    trips = spark.table(f"{catalog}.silver.trips_geospatial")

    # ------------------------------------------------------------------
    # 1️⃣ Features temporelles
    # ------------------------------------------------------------------
    features = (
        trips
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("week_of_year", F.weekofyear("tpep_pickup_datetime"))
        .withColumn("month", F.month("tpep_pickup_datetime"))
        .withColumn("year", F.year("tpep_pickup_datetime"))
        .withColumn("is_morning", F.col("hour_of_day").between(6, 10))
        .withColumn("is_evening", F.col("hour_of_day").between(16, 20))
        .withColumn(
            "is_night",
            (F.col("hour_of_day") >= 22) | (F.col("hour_of_day") <= 4)
        )
    )

    # ------------------------------------------------------------------
    # 2️⃣ Features géographiques
    # ------------------------------------------------------------------
    features = (
        features
        .withColumn(
            "is_manhattan_trip",
            (F.col("pickup_borough") == "Manhattan") &
            (F.col("dropoff_borough") == "Manhattan")
        )
    )

    # ------------------------------------------------------------------
    # 3️⃣ Features météo
    # ------------------------------------------------------------------
    features = (
        features
        .withColumn("is_rainy", F.col("prcp") > 0)
        .withColumn("is_cold", F.col("temp") < 32)   # < 0°C
        .withColumn("is_hot", F.col("temp") > 86)    # > 30°C
    )

    # ------------------------------------------------------------------
    # 4️⃣ Features dérivées business
    # ------------------------------------------------------------------
    features = (
        features
        .withColumn(
            "fare_per_mile",
            F.when(F.col("trip_distance") > 0,
                   F.col("total_amount") / F.col("trip_distance"))
        )
        .withColumn(
            "fare_per_minute",
            F.when(F.col("trip_duration_minutes") > 0,
                   F.col("total_amount") / F.col("trip_duration_minutes"))
        )
    )

    # ------------------------------------------------------------------
    # 5️⃣ Features trafic (fenêtres temporelles)
    # ------------------------------------------------------------------
    traffic_window = (
        Window
        .partitionBy("pickup_zone")
        .orderBy("tpep_pickup_datetime")
        .rowsBetween(-100, -1)
    )

    features = (
        features
        .withColumn("recent_trips_in_zone", F.count("*").over(traffic_window))
        .withColumn("avg_recent_duration", F.avg("trip_duration_minutes").over(traffic_window))
        .withColumn("avg_recent_fare", F.avg("total_amount").over(traffic_window))
    )

    # ------------------------------------------------------------------
    # 6️⃣ Sélection finale (features + targets)
    # ------------------------------------------------------------------

    ml_features = features.select(
    # 🎯 Targets
    "trip_duration_minutes",
    "total_amount",
    "tip_amount",

    # ⏱️ Temporel
    "hour_of_day",
    "day_of_week",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "is_morning",
    "is_evening",
    "is_night",

    # 🗺️ Géographie
    "pickup_borough",
    "dropoff_borough",
    "pickup_zone",
    "dropoff_zone",
    "is_same_borough",
    "is_manhattan_trip",
    "is_airport_trip",
    "trip_distance",
    "real_distance_miles",

    # 🌦️ Météo
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "is_rainy",
    "is_cold",
    "is_hot",

    # 💰 Business
    "passenger_count",
    "fare_per_mile",
    "fare_per_minute",

    # 🚦 Trafic
    "recent_trips_in_zone",
    "avg_recent_duration",
    "avg_recent_fare",

    # Metadata
    "VendorID",
    "tpep_pickup_datetime"
)

    

    # ------------------------------------------------------------------
    # 7️⃣ Écriture GOLD
    # ------------------------------------------------------------------
    ml_features.writeTo(f"{catalog}.gold.ml_trip_features") \
        .partitionedBy("year", "month") \
        .createOrReplace()

    print("✅ gold.ml_trip_features créée")

    # ------------------------------------------------------------------
    # 8️⃣ Dataset de travail (sample)
    # ------------------------------------------------------------------
    ml_sample = ml_features.sample(0.1, seed=42)

    ml_sample.writeTo(f"{catalog}.gold.ml_trip_features_sample") \
        .createOrReplace()

    print("✅ gold.ml_trip_features_sample créée")

    print("🎉 FEATURE STORE ML PRÊT")

    return ml_features


In [32]:
ml_features = create_ml_feature_store()
ml_features.printSchema()


🧠 Démarrage Feature Engineering ML


✅ gold.ml_trip_features créée


✅ gold.ml_trip_features_sample créée
🎉 FEATURE STORE ML PRÊT
root
 |-- trip_duration_minutes: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- is_morning: boolean (nullable = true)
 |-- is_evening: boolean (nullable = true)
 |-- is_night: boolean (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- is_same_borough: boolean (nullable = true)
 |-- is_manhattan_trip: boolean (nullable = true)
 |-- is_airport_trip: boolean (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- real_distance_miles: double (nullable = t

In [20]:
spark.table("iceberg.gold.ml_trip_features_sample").limit(5).toPandas()

,trip_duration_minutes,total_amount,tip_amount,hour_of_day,day_of_week,week_of_year,month,year,is_weekend,is_morning,is_evening,is_night,pickup_borough,dropoff_borough,pickup_zone,dropoff_zone,is_same_borough,is_manhattan_trip,is_airport_trip,trip_distance,real_distance_miles,temp,rhum,prcp,wspd,pres,is_rainy,is_cold,is_hot,passenger_count,fare_per_mile,fare_per_minute,recent_trips_in_zone,avg_recent_duration,avg_recent_fare,VendorID,tpep_pickup_datetime
0,11.550000,19.20,0.00,0,2,1,1,2024,False,False,False,True,Manhattan,Manhattan,Midtown East,West Village,True,True,False,2.71,2.210504,5.0,55.0,0.0,9.0,1016.0,False,True,False,3,7.084871,1.662338,3,10.583333,18.500000,2,2024-01-01 00:02:52
1,12.216667,25.40,2.00,0,2,1,1,2024,False,False,False,True,Manhattan,Manhattan,Midtown East,Yorkville West,True,True,False,3.67,1.890719,5.0,55.0,0.0,9.0,1016.0,False,True,False,1,6.920981,2.079127,21,14.623810,21.891905,2,2024-01-01 00:16:04
2,9.416667,19.68,3.28,0,2,1,1,2024,False,False,False,True,Manhattan,Manhattan,Midtown East,East Village,True,True,False,1.81,2.130517,5.0,55.0,0.0,9.0,1016.0,False,True,False,2,10.872928,2.089912,26,16.712821,24.778462,2,2024-01-01 00:19:59
3,7.450000,12.90,0.00,0,2,1,1,2024,False,False,False,True,Manhattan,Manhattan,Midtown East,Murray Hill,True,True,False,0.59,0.696303,5.0,55.0,0.0,9.0,1016.0,False,True,False,2,21.864407,1.731544,41,17.328049,25.789512,2,2024-01-01 00:27:56
4,15.733333,24.72,4.12,0,2,1,1,2024,False,False,False,True,Manhattan,Manhattan,Midtown East,Lincoln Square East,True,True,False,1.90,1.265493,5.0,55.0,0.0,9.0,1016.0,False,True,False,1,13.010526,1.571186,48,16.498611,24.944375,2,2024-01-01 00:30:23
